> 在线讲义：[第 28 章 · 编译器：表达、变换与选择](https://wikieden.github.io/ai-infra-from-constraints/lessons/28-compiler.html) · 本书《从约束推导 AI Infra》

# 第 28 章 · 编译器：表达、变换与选择

配套讲义：仓库里 `ai-infra/notes/lessons/28-compiler.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 划分 | $2^{n-1}$ | `partitions(10)` | 512 |
| 第三步 次序 | | `silu(1) + silu(-1)` | 0.4621 |
| 第三步 例 5-7 | E4M3 舍入 | `quantize_first(1, 10)`、`quantize_first(1, 1)` | 55/56、1 |
| 第四步 驻留 | 100 KB ÷（工作集 + 1 KiB） | `resident(32)`、`resident(96)` | 3 / 1 |
| 第四步 例 5-9 | $20 - 15p$ | `mix_time(p)` | $p > 2/3$ |

融合划分数、SiLU 与归约的次序、例 5-7 的 E4M3 整行 scale、双缓冲驻留、例 5-9 的调用比例取自 AII 第 5 章 §5.4。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math

**函数 `partitions`**：一条 n 个算子的链，每条相邻边界可切可不切：2^(n-1) 种。

In [ ]:
def partitions(n_ops: int) -> int:
    """一条 n 个算子的链，每条相邻边界可切可不切：2^(n-1) 种。"""
    return 2 ** (n_ops - 1)

**函数 `silu`**：小工具。

In [ ]:
def silu(x: float) -> float:
    return x / (1 + math.exp(-x))

**函数 `e4m3_round`**：舍入到 E4M3FN（3 位尾数，最大 448），就近取偶，只处理正数。

In [ ]:
def e4m3_round(x: float) -> float:
    """舍入到 E4M3FN（3 位尾数，最大 448），就近取偶，只处理正数。"""
    if x >= 448:
        return 448.0
    e = max(math.floor(math.log2(x)), -6) if x > 0 else -6
    step = 2.0 ** (e - 3)
    q = x / step
    lo = math.floor(q)
    frac = q - lo
    r = lo + (1 if frac > 0.5 or (frac == 0.5 and lo % 2) else 0)
    return r * step

**函数 `quantize_first`**：按整行最大绝对值把 first 映射到 448，舍入后反量化。

In [ ]:
def quantize_first(first: float, row_max: float) -> float:
    """按整行最大绝对值把 first 映射到 448，舍入后反量化。"""
    scale = 448 / row_max
    return e4m3_round(first * scale) / scale

**函数 `resident`**：小工具。

In [ ]:
def resident(work_kib: int, smem: int = 102_400, reserve: int = 1024) -> int:
    return smem // (work_kib * 1024 + reserve)

**函数 `mix_time`**：形状 A 占比 p 时新实现的平均时间（μs）。

In [ ]:
def mix_time(p: float, a: float = 5, b: float = 20) -> float:
    """形状 A 占比 p 时新实现的平均时间（μs）。"""
    return a * p + b * (1 - p)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：候选有多少")
print(f"  3 个算子 {partitions(3)} 种划分，10 个算子 {partitions(10)} 种（还没算块大小、布局、流水深度）")

In [ ]:
print("\n第二步：依赖与舍入")
print(f"  先加后激活 SiLU(1 − 1) = {silu(0):.4f}；先激活后加 SiLU(1) + SiLU(−1) = {silu(1) + silu(-1):.4f}")
full, early = quantize_first(1, 10), quantize_first(1, 1)
print(f"  例 5-7：读完整行再量化 {full:.6f}（= 55/56），读完第一块就量化 {early:.6f}，差 {(early - full) * 100:.1f}%")

In [ ]:
print("\n第三步：成本估计与实测选择")
print(f"  RTX PRO 6000 100 KB：单缓冲 24 / 80 KiB 驻留 {resident(24)} / {resident(80)} 块；双缓冲 32 / 96 KiB 驻留 {resident(32)} / {resident(96)} 块")
print(f"  例 5-9：各一次 20 → {5 + 20} μs；A 100 次、B 1 次 {100 * 10 + 10} → {100 * 5 + 20} μs（{1010 / 520:.1f} 倍）")
print(f"  平均时间 20 − 15p < 10 ⇔ p > {2 / 3:.3f}；p = 0.9 时 {mix_time(0.9):.1f} μs")

## 3. 代码里才看得到的

1. E4M3 在 32–64 之间步长是 4，所以 44.8 落在 44 和 48 之间、舍入到 44。
2. 指数下限 −6 对应次正规数区间；这里的例子用不到，但 `e4m3_round(0.3)` 能看到步长变成 1/32。
3. `mix_time(2/3)` 正好等于 10：交点在浮点里也是精确的（$20 - 15 \times 2/3 = 10$）。

## 4. 自己改一改

- 给 `quantize_first` 加"每 32 个元素一组 scale"的版本，例 5-7 的两块各自定 scale，误差还有吗？
- 把例 5-9 推广成三种形状，写出新实现更快的条件。
- 用 `partitions` 加约束：哪些边界必须切（归约、整行 scale），数一数剩下多少种。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 箱子怎么分组怎么装，组合多到只能比；
> 切块、换序、定在哪层算，缩进就是寿命；
> 先汇总再打折，顺序是规则的一部分；整行最大值没读到，量化别动手；
> 先筛硬条件，再估分，最后按真实活计面试。